# 2b · Manual labels for the mixed test atoms

Notebook 2 atomises whole answers. Atoms of **single-class answers** (EEEE / CCCC / NNNN) inherit the answer label. Atoms of **mixed answers** (test only) are left unlabelled: their answer mixes labels, so an atom's label cannot be inherited. This notebook has them labelled by hand.

**Design (standard for inter-annotator agreement on a large set):**
1. **Shared set:** `N_SHARED` mixed atoms that **all three annotators** label. They give the inter-annotator agreement (Fleiss' κ, Cohen's κ per pair). Sampled stratified by `gemini_label`, so the rare contradictions are well represented.
2. **The rest** of the mixed atoms is split evenly over the annotators: **one label each**.
3. **Blind sheets:** each annotator gets one Excel sheet (shared + own atoms, shuffled), showing only the premise and the atom (what the models see), plus a code (`ref`) that says nothing about the atom. Nobody sees Gemini's label, the answer's label, the drug pair or the answer the atom came from, which atoms are shared, or anyone else's labels.
4. **Import:**
   - shared atoms: the majority label (2 of 3); atoms where all three disagree go to `adjudication.csv`;
   - other atoms: the annotator's label;
   - report: κ, progress per annotator, and agreement of the human labels with `gemini_label` (how good Gemini's sentence labels were).
5. **Output:** `data/atomic_test_final.csv`, the test atoms for notebook 4: single-class atoms with their inherited label (`label_source = answer`) and mixed atoms with their human label (`label_source = human`).

Runs after notebook 2. **Do not re-run notebook 2 after exporting the sheets**: the key links each `ref` to an `atom_id`, and the import checks that the atom text still matches.

In [ ]:
SPLIT = "test"
ANNOTATORS = ["A", "B", "C"]
N_SHARED = 300                 # mixed atoms labelled by every annotator (inter-annotator agreement)
SEED = 21
OVERWRITE_SHEETS = False       # True rebuilds the sheets and the key - this discards filled-in labels
AUDIT_DIR = "data/audit"
LABELS = ["entailment", "contradiction", "neutral"]
KAPPA_BOOTSTRAP = 1000         # resamples for the 95% confidence intervals of kappa (0 = no intervals)

## Load the test atoms

In [ ]:
import hashlib, math, os
from itertools import combinations

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import cohen_kappa_score

os.makedirs(AUDIT_DIR, exist_ok=True)
KEY_PATH = f"{AUDIT_DIR}/audit_key.csv"
ADJ_PATH = f"{AUDIT_DIR}/adjudication.csv"
FINAL_PATH = f"data/atomic_{SPLIT}_final.csv"

atoms = pd.read_csv(f"data/atomic_{SPLIT}.csv")
needed = {"atom_id", "label_source", "gemini_label", "gen_mode"}
if needed - set(atoms.columns):
    raise ValueError(f"data/atomic_{SPLIT}.csv has no {sorted(needed - set(atoms.columns))}: run the current notebook 2 "
                     f"(ATOMISE_FROM = 'answer') first.")
assert atoms["atom_id"].is_unique, "atom_id is not unique: re-run notebook 2"


def ref_of(atom_id):
    # The code shown on the sheets. atom_id holds the answer's label, so it must not reach the annotators.
    return hashlib.sha1(str(atom_id).encode()).hexdigest()[:8]


atoms["ref"] = atoms["atom_id"].map(ref_of)
assert atoms["ref"].is_unique
manual = atoms[atoms["label_source"] == "manual"]
print(f"{len(atoms):,} {SPLIT} atoms: {(atoms['label_source'] == 'answer').sum():,} with an inherited label, "
      f"{len(manual):,} mixed atoms to label by hand")
print("mixed atoms by Gemini's sentence label:", manual["gemini_label"].value_counts().to_dict())
n_rest = len(manual) - min(N_SHARED, len(manual))
print(f"workload: {min(N_SHARED, len(manual))} shared + about {math.ceil(n_rest / len(ANNOTATORS))} own atoms per annotator")

## Export the annotation sheets

**Annotation guideline (standard NLI).** Read the premise, then the atom, and write one label in `label`. Judge the atom **as it is written**, against the premise only, the same two texts the models see (the sheet does not say which drug pair an answer was about, because the models are not told either).
- **entailment (e):** if the premise is true, the atom must be true.
- **contradiction (c):** if the premise is true, the atom must be false.
- **neutral (n):** anything else.

How the usual cases fall:
- **Dropped condition** (a patient group, a dose limit, the drug the interaction is with) → neutral.
- **Added detail** the premise does not give (dose, timing, mechanism, population, route) → neutral.
- **Hedges:** "may increase" → "increases" is neutral; "increases" → "may increase" is entailment.
- **"Some" → "all"** → neutral.
- **Two drugs named together:** a claim about each drug needs the premise to say it about each; a claim about using them together ("combined", "concomitant", "the combination of") needs the premise to say it about that combination.
- **Different measure** (metabolism vs excretion, bioavailability vs clearance) → neutral; **opposite direction** on the same measure → contradiction.
- **Denying what the premise says can happen** ("may potentiate" → "does not potentiate") → contradiction.
- **Swapping which drug affects which** → neutral.

Use `unsure` (y) for hard cases and `notes` for anything else. Judge against the premise only, without looking anything up, and without discussing atoms with the other annotators until the import is done.

The sheets are written once. Re-running this cell leaves existing sheets alone unless `OVERWRITE_SHEETS = True`.

In [ ]:
from openpyxl.styles import Alignment, Font
from openpyxl.worksheet.datavalidation import DataValidation

WIDTHS = {"item": 6, "ref": 10, "premise": 80, "atom": 60, "label": 15, "unsure": 8, "notes": 30}


def write_sheet(sheet, path):
    # Excel sheet: wrapped text, frozen header row, a dropdown (entailment / contradiction / neutral) in `label`
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        sheet.to_excel(writer, index=False, sheet_name="labels")
        ws = writer.sheets["labels"]
        for i, col in enumerate(sheet.columns, 1):
            letter = ws.cell(row=1, column=i).column_letter
            ws.column_dimensions[letter].width = WIDTHS.get(col, 12)
            ws.cell(row=1, column=i).font = Font(bold=True)
            for (cell,) in ws.iter_rows(min_row=2, min_col=i, max_col=i):
                cell.alignment = Alignment(wrap_text=True, vertical="top")
        ws.freeze_panes = "A2"
        letter = ws.cell(row=1, column=list(sheet.columns).index("label") + 1).column_letter
        dv = DataValidation(type="list", formula1='"entailment,contradiction,neutral"', allow_blank=True)
        ws.add_data_validation(dv)
        dv.add(f"{letter}2:{letter}{len(sheet) + 1}")


def export_sheets():
    pool = manual.sample(frac=1, random_state=SEED)
    # shared set: as equal as possible over Gemini's labels (smallest group first; what a small group cannot fill
    # goes to the larger ones), so the rare contradictions are well represented
    target = min(N_SHARED, len(pool))
    groups = sorted(pool.groupby("gemini_label"), key=lambda kv: len(kv[1]))
    shared_ids = []
    for i, (_, g) in enumerate(groups):
        quota = math.ceil((target - len(shared_ids)) / (len(groups) - i))
        shared_ids += list(g["atom_id"][:quota])
    shared = set(shared_ids)
    # the rest: round-robin per Gemini label, so every annotator gets the same label mix
    rest = pool[~pool["atom_id"].isin(shared)].sort_values("gemini_label", kind="stable")
    owner = {aid: ANNOTATORS[i % len(ANNOTATORS)] for i, aid in enumerate(rest["atom_id"])}
    key = manual.assign(assigned_to=manual["atom_id"].map(lambda a: "ALL" if a in shared else owner[a]))
    key.to_csv(KEY_PATH, index=False)
    for k, annotator in enumerate(ANNOTATORS):
        mine = key[key["assigned_to"].isin(["ALL", annotator])]
        # premise and atom only: the models do not see the drug pair either
        sheet = mine[["ref", "premise", "hypothesis"]].rename(columns={"hypothesis": "atom"})
        sheet = sheet.sample(frac=1, random_state=SEED + 1 + k).assign(label="", unsure="", notes="")
        sheet.insert(0, "item", range(1, len(sheet) + 1))
        path = f"{AUDIT_DIR}/audit_sheet_{annotator}.xlsx"
        write_sheet(sheet, path)
        old = path[:-5] + ".csv"                 # earlier export, which showed atom_id
        if os.path.exists(old):
            os.remove(old)
        print(f"{path}: {len(sheet)} atoms")
    print(f"shared by all: {len(shared)} | split: {len(rest)} | keep {KEY_PATH} away from the annotators")
    display(key.groupby(["assigned_to", "gemini_label"]).size().unstack(fill_value=0))


if os.path.exists(KEY_PATH) and not OVERWRITE_SHEETS:
    print(f"{KEY_PATH} already exists: the sheets were exported before and may be filled in, so they are left alone. "
          f"Set OVERWRITE_SHEETS = True to rebuild them (this discards their labels).")
else:
    export_sheets()

## Import the annotations

1. **Read** the filled sheets (`audit_sheet_{A,B,C}.xlsx`, or `.csv`). Labels may be written as `e`, `c` or `n`, or in full. Rows are matched to atoms by `ref`, so sorting or filtering a sheet is fine. The atom text is checked against the key, so a sheet from an older notebook 2 run is noticed.
2. **Agreement** on the shared atoms: Cohen's κ for every pair as soon as both annotators have labelled them, Fleiss' κ once all three have, each with a 95% bootstrap interval and its Landis & Koch strength, plus a confusion table per pair; saved to `iaa_report.csv`.
3. **Gold label:** shared atoms take the majority label; three-way splits go to `adjudication.csv` (fill `final_label`, run again). Other atoms take their annotator's label.
4. **Report:** progress per annotator, and how often the human label agrees with `gemini_label`, per label.
5. **Write** `data/atomic_test_final.csv`. Mixed atoms without a label yet are left out of it, and the report says so.

In [ ]:
ALIASES = {"e": "entailment", "ent": "entailment", "entail": "entailment", "entailment": "entailment",
           "c": "contradiction", "con": "contradiction", "contra": "contradiction", "contradiction": "contradiction",
           "n": "neutral", "neu": "neutral", "neutral": "neutral"}


def read_sheet(annotator, key):
    paths = [f"{AUDIT_DIR}/audit_sheet_{annotator}.{ext}" for ext in ("xlsx", "csv")]
    path = next((p for p in paths if os.path.exists(p)), None)
    if path is None:
        return pd.Series(dtype=object)
    if path.endswith(".xlsx"):
        sheet = pd.read_excel(path, dtype=str)
    else:
        # sep=None: also reads sheets saved by Excel with ";" (Indonesian / European regional settings); utf-8-sig: BOM
        sheet = pd.read_csv(path, dtype=str, sep=None, engine="python", encoding="utf-8-sig")
    sheet = sheet.fillna("")
    if "ref" in sheet.columns:
        atom_of = pd.Series(key.index, index=key["ref"].values)
        unknown = ~sheet["ref"].isin(atom_of.index)
        if unknown.any():
            print(f"[{annotator}] WARNING: {int(unknown.sum())} rows with an unknown ref: ignored")
        sheet = sheet[~unknown].copy()
        sheet["atom_id"] = sheet["ref"].map(atom_of).values
    sheet = sheet.set_index("atom_id")      # sheets exported before the ref column carry atom_id themselves
    changed = sheet.index.intersection(key.index)
    changed = changed[sheet.loc[changed, "atom"] != key.loc[changed, "hypothesis"]]
    if len(changed):
        print(f"[{annotator}] WARNING: {len(changed)} atoms differ from the key (notebook 2 re-run?): ignored")
    raw = sheet["label"].str.strip().str.lower().drop(changed)
    labels = raw.map(ALIASES)
    bad = sorted(set(raw[(raw != "") & labels.isna()]))
    if bad:
        print(f"[{annotator}] unreadable labels ignored: {bad}")
    return labels


def kappa(a, b):
    return cohen_kappa_score(a, b) if len(a) and (pd.Series(a).nunique() > 1 or pd.Series(b).nunique() > 1) else float("nan")


def fleiss_kappa(votes):
    r = votes.shape[1]
    counts = np.stack([(votes == lab).sum(axis=1).to_numpy() for lab in LABELS], axis=1)
    if len(counts) == 0 or r < 2:
        return float("nan")
    p_atom = ((counts ** 2).sum(axis=1) - r) / (r * (r - 1))
    p_label = counts.sum(axis=0) / counts.sum()
    p_chance = (p_label ** 2).sum()
    return float("nan") if p_chance == 1 else (p_atom.mean() - p_chance) / (1 - p_chance)


def strength(k):
    # Landis & Koch (1977)
    if pd.isna(k):
        return ""
    if k < 0:
        return "poor"
    for upper, name in ((0.20, "slight"), (0.40, "fair"), (0.60, "moderate"), (0.80, "substantial")):
        if k <= upper:
            return name
    return "almost perfect"


def bootstrap_ci(frame, stat):
    # 95% percentile interval over atoms resampled with replacement
    if KAPPA_BOOTSTRAP <= 0 or len(frame) < 10:
        return float("nan"), float("nan")
    rng = np.random.default_rng(SEED)
    vals = [stat(frame.iloc[rng.integers(0, len(frame), len(frame))]) for _ in range(KAPPA_BOOTSTRAP)]
    vals = [v for v in vals if pd.notna(v)]
    return tuple(np.percentile(vals, [2.5, 97.5])) if vals else (float("nan"), float("nan"))


def agreement_report(votes, key):
    """Cohen's kappa for every pair (on the shared atoms both labelled) and Fleiss' kappa (on those all labelled)."""
    shared = votes[key["assigned_to"].eq("ALL").reindex(votes.index)]
    rows, pairs = [], []
    for a, b in combinations(ANNOTATORS, 2):
        both = shared[[a, b]].dropna()
        k = kappa(both[a], both[b]) if len(both) else float("nan")
        lo, hi = bootstrap_ci(both, lambda f: kappa(f[a], f[b]))
        rows.append({"measure": f"Cohen's kappa {a}-{b}", "atoms": len(both),
                     "raw_agreement": (both[a] == both[b]).mean() if len(both) else float("nan"),
                     "kappa": k, "ci95_low": lo, "ci95_high": hi, "strength": strength(k)})
        if len(both):
            pairs.append((a, b, both))
    full = shared.dropna()
    if len(full) and len(ANNOTATORS) > 2:
        k = fleiss_kappa(full)
        lo, hi = bootstrap_ci(full, fleiss_kappa)
        rows.append({"measure": f"Fleiss' kappa ({', '.join(ANNOTATORS)})", "atoms": len(full),
                     "raw_agreement": full.nunique(axis=1).eq(1).mean(),
                     "kappa": k, "ci95_low": lo, "ci95_high": hi, "strength": strength(k)})
    table = pd.DataFrame(rows).set_index("measure")
    print(f"\nINTER-ANNOTATOR AGREEMENT on the {int(key['assigned_to'].eq('ALL').sum())} shared atoms")
    display(table.round(3))
    print("raw_agreement: share of atoms with the same label (Fleiss: all annotators the same); "
          "ci95: bootstrap over atoms; strength: Landis & Koch (1977)")
    for a, b, both in pairs:
        print(f"\n{a} (rows) vs {b} (columns), {len(both)} shared atoms:")
        display(pd.crosstab(both[a].rename(a), both[b].rename(b)))
    table.to_csv(f"{AUDIT_DIR}/iaa_report.csv")
    print(f"saved {AUDIT_DIR}/iaa_report.csv")
    return table


def import_annotations():
    key = pd.read_csv(KEY_PATH).set_index("atom_id")
    key["ref"] = [ref_of(a) for a in key.index]          # also for keys written before the ref column
    votes = pd.DataFrame({a: read_sheet(a, key) for a in ANNOTATORS}).reindex(key.index)

    print("progress:")
    for a in ANNOTATORS:
        assigned = key["assigned_to"].isin(["ALL", a])
        print(f"  {a}: {int(votes.loc[assigned, a].notna().sum())}/{int(assigned.sum())} labelled")

    agreement_report(votes, key)

    shared = key["assigned_to"].eq("ALL")
    full = votes[shared & votes.notna().all(axis=1)]
    print(f"\nGOLD LABELS\nshared atoms labelled by all {len(ANNOTATORS)}: {len(full)}/{int(shared.sum())}")
    gold = pd.Series([None] * len(key), index=key.index, dtype=object)
    n_agree = pd.Series(np.nan, index=key.index)
    if len(full):
        top = full.apply(lambda row: row.value_counts().iloc[0], axis=1)
        print("annotators agreeing:", ", ".join(f"{k}/{len(ANNOTATORS)}: {v}" for k, v in
                                               top.value_counts().sort_index(ascending=False).items()))
        n_agree[full.index] = top
        majority = full.apply(lambda row: row.value_counts().index[0], axis=1).where(top > len(ANNOTATORS) / 2)
        gold.loc[majority.dropna().index] = majority.dropna().astype(object)

    split = full.index[gold[full.index].isna()]
    previous = (pd.read_csv(ADJ_PATH, dtype=str).fillna("").set_index("atom_id")["final_label"]
                if os.path.exists(ADJ_PATH) else pd.Series(dtype=str))
    adjudicated = previous.str.strip().str.lower().map(ALIASES).dropna()
    adjudicated = adjudicated[adjudicated.index.isin(split)]
    if len(adjudicated):
        gold.loc[adjudicated.index] = adjudicated.astype(object)
    if len(split):
        sheet = key.loc[split, ["premise", "hypothesis"]].join(votes.loc[split])
        sheet["final_label"] = previous.reindex(split).fillna("")
        sheet.reset_index().to_csv(ADJ_PATH, index=False)
        open_cases = int((sheet["final_label"].str.strip() == "").sum())
        print(f"{len(split)} three-way splits in {ADJ_PATH}"
              + (f": {open_cases} still need a final_label" if open_cases else ", all adjudicated"))

    single = ~shared
    owner_vote = pd.Series([votes.at[i, key.at[i, "assigned_to"]] if not s else np.nan
                            for i, s in zip(key.index, shared)], index=key.index)
    gold.loc[single] = owner_vote[single].astype(object)
    n_agree[single & gold.notna()] = 1

    done = gold.notna()
    print(f"\nmixed atoms with a human label: {int(done.sum())}/{len(key)}")
    if done.any():
        labelled = key[done].assign(human=gold[done])
        labelled["agree"] = labelled["human"] == labelled["gemini_label"]
        print(f"human label = Gemini's sentence label: {labelled['agree'].mean():.1%} "
              f"(kappa {kappa(labelled['human'], labelled['gemini_label']):.3f})")
        display(labelled.groupby("gemini_label")["agree"].agg(["mean", "count"]).round(3))
        display(pd.crosstab(labelled["gemini_label"].rename("gemini"), labelled["human"].rename("human")))

    final = atoms.set_index("atom_id")
    final.loc[gold[done].index, "label"] = gold[done]
    final.loc[gold[done].index, "label_source"] = "human"
    final["human_votes"] = votes.apply(lambda r: ",".join(f"{a}:{r[a][0]}" for a in ANNOTATORS if pd.notna(r[a])),
                                       axis=1).reindex(final.index)
    final["n_agree"] = n_agree.reindex(final.index)
    missing = final["label"].isna()
    if missing.any():
        print(f"PROVISIONAL: {int(missing.sum())} mixed atoms have no label yet and are left out of {FINAL_PATH}")
    out = final[~missing].reset_index()
    out.to_csv(FINAL_PATH, index=False)
    print(f"wrote {FINAL_PATH}: {len(out):,} atoms ({out['label_source'].value_counts().to_dict()})")


if os.path.exists(KEY_PATH):
    import_annotations()
else:
    print(f"{KEY_PATH} not found: export the sheets first")